# Round-1 Annotation Pipeline — Kaggle Runner

**Model**: `Qwen/Qwen3.5-2B` (natively multimodal, 2B params, ~2 GB VRAM in 4-bit)  
**GPU**: T4 — 16 GB VRAM (Kaggle free tier)

## Before running — checklist

| # | Step | Where |
|---|------|-------|
| 1 | Enable **GPU T4** | Session options → Accelerator → GPU T4 |
| 2 | Enable **Internet** | Session options → Internet → On |
| 3 | Add `HF_TOKEN` secret | Add-ons → Secrets → New secret |
| 4 | Attach your dataset | Add data → Your datasets |
| 5 | Set `REPO_URL` and `DATASET_SLUG` in **Cell 4** | — |

> **Internet must be ON** — the notebook clones the repo from GitHub and downloads  
> the model weights (~4 GB) from HuggingFace on first run.

The pipeline saves a checkpoint after every batch — re-running the notebook resumes from where it stopped.

## 1. Install dependencies

`Qwen3.5-2B` requires `transformers >= 4.51.0`. We install from the GitHub source to get the latest stable version.

In [ ]:
import subprocess, sys

def _pip(*args):
    """Run pip and print output; raise on failure."""
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:] if r.stdout else "")
        raise RuntimeError(f"pip install failed:\n{r.stderr[-2000:]}")

_pip("git+https://github.com/huggingface/transformers")  # >= 4.51.0 for Qwen3.5-2B
_pip("accelerate>=0.30.0", "bitsandbytes>=0.43.0",
     "pydantic>=2.0", "pyyaml", "tqdm", "Pillow")
print("All dependencies installed.")

## 2. Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU in Session options → Accelerator → GPU T4.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"Disk free (/kaggle/working) : {free_gb:.1f} GB")
if free_gb < 6:
    print("WARNING: Less than 6 GB free — model download (~4.2 GB) may fail.")

## 3. Verify Qwen3.5-2B transformers support

In [ ]:
import transformers
print(f"Transformers version : {transformers.__version__}")

try:
    from transformers import AutoModelForImageTextToText
    print("AutoModelForImageTextToText : OK")
except ImportError as e:
    raise RuntimeError(
        f"AutoModelForImageTextToText not found: {e}\n"
        "Re-run Cell 1 to reinstall transformers from source."
    )

## 4. Environment configuration

**Edit these two variables before running:**

In [ ]:
import os
from pathlib import Path

# GitHub repo URL — set to None if code is already present in /kaggle/working/
REPO_URL = "https://github.com/YOUR_USERNAME/social-media-mining.git"

# Kaggle dataset slug (the folder name under /kaggle/input/)
# e.g. if your dataset is at /kaggle/input/vn-sarcasm-data/ → "vn-sarcasm-data"
DATASET_SLUG = "vn-sarcasm-data"

# ---------------------------------------------------------------------------
# Derived paths — do not edit below this line
# ---------------------------------------------------------------------------
KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_INPUT   = Path("/kaggle/input")
REPO_DIR       = KAGGLE_WORKING / "vi-multimodal-sacarsm-detection-on-social-media"
ROUND1_DIR     = REPO_DIR / "round-1-annotation"
DATASET_DIR    = KAGGLE_INPUT / DATASET_SLUG
OUTPUT_DIR     = KAGGLE_WORKING / "outputs"

# Redirect HuggingFace cache to /kaggle/working to avoid filling /root/.cache
HF_CACHE_DIR = KAGGLE_WORKING / "hf_cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)

print(f"Dataset dir  : {DATASET_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"HF cache     : {HF_CACHE_DIR}")

if not DATASET_DIR.exists():
    print(f"WARNING: {DATASET_DIR} not found. Attach your dataset in Add data → Your datasets.")

## 5. Clone repo and set Python path

In [ ]:
import subprocess, sys

if REPO_URL and not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} ...")
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{r.stderr}")
    print("Clone complete.")
elif REPO_DIR.exists():
    print(f"Repo present at {REPO_DIR} — pulling latest ...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    print("REPO_URL is None — using code already in REPO_DIR.")

if str(ROUND1_DIR) not in sys.path:
    sys.path.insert(0, str(ROUND1_DIR))
print(f"sys.path includes: {ROUND1_DIR}")

## 6. Load HuggingFace token from Kaggle Secrets

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment variable.")
    else:
        print("HF_TOKEN not set — will proceed without it (public models only).")

## 7. Locate input data

In [ ]:
import json

INPUT_JSON = DATASET_DIR / "data.json"
if not INPUT_JSON.exists():
    for candidate in ["data.jsonl", "data-sample.json", "annotations.json"]:
        p = DATASET_DIR / candidate
        if p.exists():
            INPUT_JSON = p
            break

if not INPUT_JSON.exists():
    raise FileNotFoundError(
        f"No input JSON found in {DATASET_DIR}.\n"
        "Set INPUT_JSON manually to the correct path."
    )

raw = INPUT_JSON.read_text(encoding="utf-8").strip()
n_records = len(json.loads(raw)) if raw.startswith("[") else sum(1 for l in raw.splitlines() if l.strip())
print(f"Input file : {INPUT_JSON}")
print(f"Records    : {n_records}")

## 8. Symlink dataset images into repo `data/` directory

Image paths in the JSON are relative to the repo root (e.g. `data/images/001.jpg`).  
Symlinking the Kaggle dataset folder into `data/` makes those paths resolve correctly without copying any files.

In [ ]:
DATA_LINK = REPO_DIR / "data"

if DATA_LINK.is_symlink():
    print(f"Symlink already exists: {DATA_LINK} -> {DATA_LINK.resolve()}")
elif DATA_LINK.exists():
    print(f"{DATA_LINK} is a real directory — skipping symlink.")
else:
    DATA_LINK.symlink_to(DATASET_DIR)
    print(f"Created symlink: {DATA_LINK} -> {DATASET_DIR}")

# Change cwd to repo root so relative paths in the pipeline resolve correctly
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 9. Verify prompt files

In [ ]:
for fname in ["prompt.txt", "few-short-examples.txt"]:
    p = ROUND1_DIR / "prompts" / fname
    status = "OK" if p.exists() else "MISSING"
    print(f"{fname:30s} : {status}")
    if not p.exists():
        raise FileNotFoundError(f"Prompt file missing: {p}")

## 10. Quick model smoke-test (optional)

Load `Qwen3.5-2B`, run one text inference, then free GPU memory.  
**Skip this cell** if you want to go straight to the pipeline and save session time.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.5-2B"
print(f"Loading {MODEL_ID} in 4-bit (this downloads ~2 GB on first run) ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
smoke_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN or None,
)
smoke_proc = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN or None)

test_msgs = [{"role": "user", "content": [{"type": "text", "text": "Reply with the single word: OK"}]}]
inputs = smoke_proc.apply_chat_template(
    test_msgs, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt"
)
# Move all tensor values to the model's device
device = next(smoke_model.parameters()).device
inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

with torch.no_grad():
    out_ids = smoke_model.generate(**inputs, max_new_tokens=10, do_sample=False)

trimmed = out_ids[0][inputs["input_ids"].shape[-1]:]
reply = smoke_proc.decode(trimmed, skip_special_tokens=True)
print(f"Smoke-test reply: '{reply}'")

del smoke_model, smoke_proc, inputs, out_ids
torch.cuda.empty_cache()
print("Model unloaded — GPU memory released for pipeline.")

## 11. Run the annotation pipeline

In [ ]:
from src.pipeline_round1 import run_pipeline

CONFIG_PATH = str(ROUND1_DIR / "configs" / "round1.yaml")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Round-1 annotation pipeline")
print(f"  Model  : Qwen/Qwen3.5-2B (4-bit)")
print(f"  Input  : {INPUT_JSON} ({n_records} records)")
print(f"  Config : {CONFIG_PATH}")
print(f"  Output : {OUTPUT_DIR}")
print("=" * 60)

run_pipeline(
    input_data=str(INPUT_JSON),
    config_path=CONFIG_PATH,
    output_dir=str(OUTPUT_DIR),
    hf_token=HF_TOKEN or None,
)

print("Pipeline finished.")

## 12. Inspect results

In [ ]:
stats_file = OUTPUT_DIR / "round1_stats.json"
if stats_file.exists():
    print(json.dumps(json.loads(stats_file.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))
else:
    print(f"Stats file not found: {stats_file}")

In [ ]:
for label, fname in [("Auto-accepted", "round1_auto_accepted.jsonl"),
                     ("Human-queue",   "round1_human_queue.jsonl")]:
    f = OUTPUT_DIR / fname
    if not f.exists():
        print(f"{label}: file not found")
        continue
    lines = f.read_text(encoding="utf-8").splitlines()
    print(f"\n{label}: {len(lines)} records")
    for line in lines[:3]:
        rec = json.loads(line)
        print(f"  id={rec['id']} | label={rec['label_llm1']} | difficulty={rec['difficulty']} | route={rec['route_reason']}")

## 13. Output file summary

Files are in `/kaggle/working/outputs/` and are committed automatically when you save the notebook run.

In [ ]:
print("Output files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.name.startswith("."):
        continue
    print(f"  {f.name:45s}  {f.stat().st_size / 1024:8.1f} KB")